## Load and Inspect the Dataset

This section loads the training and test datasets, parses the observation timestamps as datetime values, and performs a quick initial inspection. We check the dataset dimensions, chronological ordering of the train/test split, and the number of missing values in each training column.

## 1. Setup and a look at the data

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb

# Folder containing the competition files.
UPLOADS = "."

# RMSE evaluation function
rmse = lambda y, p: float(
    np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2))
)

# Load the labelled training data.
tr = pd.read_csv(
    f"{UPLOADS}/train.csv",
    parse_dates=["observation_timestamp"]
)

# Load the hidden test data.
te = pd.read_csv(
    f"{UPLOADS}/test.csv",
    parse_dates=["observation_timestamp"]
)

# Check the size of each dataset.
print(tr.shape, te.shape)

# Check the time period covered by the training set.
print(
    "train:",
    tr.observation_timestamp.min(),
    "->",
    tr.observation_timestamp.max()
)
print(
    "test :",
    te.observation_timestamp.min(),
    "->",
    te.observation_timestamp.max()
)

# Inspect missing values in the training data.
print(tr.isna().sum())

(360954, 19) (51063, 18)
train: 2013-03-01 00:00:00 -> 2016-08-31 22:00:00
test : 2016-08-31 23:00:00 -> 2017-02-28 22:00:00
id                           0
observation_timestamp        0
station                      0
year                         0
month                        0
day                          0
hour                         0
PM10                      1922
SO2                       4663
NO2                       7521
CO                       15832
O3                        8499
TEMP                       177
PRES                       178
DEWP                       182
RAIN                       174
wd                         793
WSPM                       175
PM2_5_next_hour              0
dtype: int64


Train ends 2016-08-31 22:00 and test starts 2016-08-31 23:00 — one continuous
hourly timeline, so features can be built straight across the join.

`PM10_at_h` (0.866) beats PM10 at the observation hour (0.848); same for CO.
`PM25_at_t` is 0.968 but unavailable at test time — which is why the model leans
on the covariate window around `h` instead of on persistence.

`PM10_at_h` (0.866) beats PM10 at the observation hour (0.848); same for CO.
`PM25_at_t` is 0.968 but unavailable at test time — which is why the model leans
on the covariate window around `h` instead of on persistence.

## 2. Feature Engineering

This section converts the raw hourly station data into a larger set of features for the PM2.5 model.

For each observation at time `t`, the prediction target is PM2.5 one hour later at `h = t + 1`. The data are first placed onto a complete hourly timeline for every station so that time shifts correspond to actual hours, even when some observations are missing.

The features describe the pollution and weather conditions around the target hour, recent trends, rolling averages, city-wide pollution levels, differences between a station and the wider monitoring network, pollutant relationships, seasonal patterns, wind conditions and missing measurements.

The model can also use predictor measurements from the target hour and nearby hours because these measurements are present in the supplied test data.

In [ ]:
"""Build the feature set used for PM2.5 next-hour prediction.

For a row observed at time t, the target is PM2.5 at h = t + 1.
The features combine measurements around h with recent history,
city-wide pollution conditions, weather and calendar information.
"""

import numpy as np
import pandas as pd

# Folder containing the train and test CSV files
UPLOADS = "."

# Pollutant measurements
POLL = ["PM10", "SO2", "NO2", "CO", "O3"]

# Weather measurements
MET = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

# All numerical sensor variables
NUMS = POLL + MET

# Convert compass wind directions into angles
WD_DEG = {
    "N": 0, "NNE": 22.5, "NE": 45, "ENE": 67.5, "E": 90, "ESE": 112.5,
    "SE": 135, "SSE": 157.5, "S": 180, "SSW": 202.5, "SW": 225,
    "WSW": 247.5, "W": 270, "WNW": 292.5, "NW": 315, "NNW": 337.5,
}


def load():
    """Load train and test data and mark which rows belong to the test set."""

    tr = pd.read_csv(
        f"{UPLOADS}/train.csv",
        parse_dates=["observation_timestamp"]
    )

    te = pd.read_csv(
        f"{UPLOADS}/test.csv",
        parse_dates=["observation_timestamp"]
    )

    # Keep track of train and test rows after the datasets are combined
    tr["is_test"] = 0
    te["is_test"] = 1

    # Test targets are unknown, so create an empty target column
    te["PM2_5_next_hour"] = np.nan

    return tr, te


def build_grid(tr, te):

    # Combine train and test because they form one continuous timeline
    full = pd.concat([tr, te], ignore_index=True)

    stations = sorted(full.station.unique())

    # Create every hour covered by the combined dataset
    hours = pd.date_range(
        full.observation_timestamp.min(),
        full.observation_timestamp.max(),
        freq="h"
    )

    # Create every possible station and hour combination
    grid = pd.MultiIndex.from_product(
        [stations, hours],
        names=["station", "ts"]
    ).to_frame(index=False)

    # Rename the timestamp so it matches the hourly grid
    full = full.rename(columns={"observation_timestamp": "ts"})

    # Attach the real observations to the complete timeline
    grid = grid.merge(
        full,
        on=["station", "ts"],
        how="left"
    )

    grid = grid.sort_values(
        ["station", "ts"]
    ).reset_index(drop=True)

    # Convert wind speed and direction into two numerical wind components
    deg = grid.wd.map(WD_DEG)
    rad = np.deg2rad(deg)

    grid["wu"] = -grid.WSPM * np.sin(rad)
    grid["wv"] = -grid.WSPM * np.cos(rad)

    # Record how many numerical measurements are missing at each station-hour
    grid["n_missing"] = grid[NUMS].isna().sum(axis=1)

    return grid

# It fills short missing gaps in each station’s time series by interpolating nearby values, creating cleaned versions of each selected column for later lag and rolling features.
def interpolate(grid, cols, limit=6):

    g = grid.groupby("station", sort=False)
    out = {}

    for c in cols:
        s = g[c].transform(
            lambda x: x.interpolate(
                limit=limit,
                limit_direction="both"
            )
        )

        out[c + "_i"] = s.astype("float32")

    return pd.DataFrame(out, index=grid.index)


def make_features(grid, offsets=(-4, -3, -2, -1, 0, 1, 2)):
    # All time offsets are measured relative to the target hour h.
    # For an observation at time t:
    # offset -1 is the observation hour t
    # offset  0 is the target hour h = t + 1
    # offset  1 is one hour after the target


    # Include the numerical sensors and the two wind components
    vcols = NUMS + ["wu", "wv"]

    # Fill short gaps before constructing time-based features
    ip = interpolate(grid, vcols)
    grid = pd.concat([grid, ip], axis=1)

    icols = [c + "_i" for c in vcols]

    # All time operations are performed separately for each station
    g = grid.groupby("station", sort=False)

    feats = {}

    # Measurements from several hours around the target hour
    for c in icols:

        for k in offsets:
            # The target hour is one hour after the row timestamp.
            # This shift retrieves the measurement at h + k.
            feats[f"{c}_o{k}"] = (
                g[c]
                .shift(-(k + 1))
                .astype("float32")
            )

        # Recent changes in each measurement
        feats[f"{c}_d1"] = (
            feats[f"{c}_o0"] - feats[f"{c}_o-1"]
        )

        feats[f"{c}_d3"] = (
            feats[f"{c}_o0"] - feats[f"{c}_o-3"]
        )

        # Change immediately after the target hour
        feats[f"{c}_dfwd"] = (
            feats[f"{c}_o1"] - feats[f"{c}_o0"]
        )

    # Recent history leading into the target hour
    for c in [
        "PM10_i", "CO_i", "NO2_i", "SO2_i",
        "O3_i", "WSPM_i", "TEMP_i", "DEWP_i"
    ]:

        # Align the measurement at the target hour with the current row
        shifted = g[c].shift(-1)
        s = shifted.groupby(grid.station, sort=False)

        # Recent average conditions over different time periods
        for w in (6, 12, 24):
            feats[f"{c}_rm{w}"] = s.transform(
                lambda x, w=w: x.rolling(
                    w,
                    min_periods=2
                ).mean()
            ).astype("float32")

        # Recent variability
        feats[f"{c}_rs12"] = s.transform(
            lambda x: x.rolling(
                12,
                min_periods=3
            ).std()
        ).astype("float32")

        # Highest value during the previous 12 hours
        feats[f"{c}_rmax12"] = s.transform(
            lambda x: x.rolling(
                12,
                min_periods=3
            ).max()
        ).astype("float32")

        # Compare the current target-hour value with its 24-hour average
        feats[f"{c}_vs_rm24"] = (
            feats[f"{c}_o0"] - feats[f"{c}_rm24"]
        )

    F = pd.DataFrame(feats, index=grid.index)

    # Build city-wide averages for each timestamp
    net_src = pd.DataFrame({"ts": grid.ts})

    for c in [
        "PM10_i", "CO_i", "NO2_i", "SO2_i", "O3_i",
        "WSPM_i", "TEMP_i", "PRES_i", "DEWP_i"
    ]:
        net_src[c] = grid[c]

    net = net_src.groupby("ts").mean()
    net.columns = ["net_" + c for c in net.columns]

    # City-wide measurements at the target hour
    net_at_h = net.reindex(
        grid.ts + pd.Timedelta("1h")
    ).to_numpy(dtype="float32")

    # City-wide measurements at the observation hour
    net_at_h_prev = net.reindex(
        grid.ts
    ).to_numpy(dtype="float32")

    for j, name in enumerate(net.columns):

        F[name] = net_at_h[:, j]

        # How much the city-wide measurement changed during the previous hour
        F[name + "_d1"] = (
            net_at_h[:, j] - net_at_h_prev[:, j]
        )

    # Compare each station with the city-wide conditions
    for c in ["PM10_i", "CO_i", "NO2_i", "O3_i"]:

        F[f"anom_{c}"] = (
            F[f"{c}_o0"] - F[f"net_{c}"]
        )

        F[f"ratio_{c}"] = (
            F[f"{c}_o0"] /
            (F[f"net_{c}"] + 1e-3)
        )

    # Measure how spread out PM10 and CO are across the monitoring network
    net_std = (
        net_src
        .groupby("ts")[["PM10_i", "CO_i"]]
        .std()
    )

    ns = net_std.reindex(
        grid.ts + pd.Timedelta("1h")
    ).to_numpy(dtype="float32")

    F["net_PM10_std"] = ns[:, 0]
    F["net_CO_std"] = ns[:, 1]

    # Ratios between pollutants can help distinguish different pollution conditions
    F["PM10_over_CO"] = (
        F["PM10_i_o0"] /
        (F["CO_i_o0"] + 1.0)
    )

    F["NO2_over_CO"] = (
        F["NO2_i_o0"] /
        (F["CO_i_o0"] + 1.0)
    )

    F["SO2_over_NO2"] = (
        F["SO2_i_o0"] /
        (F["NO2_i_o0"] + 1.0)
    )

    F["O3_over_NO2"] = (
        F["O3_i_o0"] /
        (F["NO2_i_o0"] + 1.0)
    )

    # Temperature minus dew point gives a simple indication of humidity
    F["dewp_depress"] = (
        F["TEMP_i_o0"] - F["DEWP_i_o0"]
    )

    F["dewp_depress_net"] = (
        F["net_TEMP_i"] - F["net_DEWP_i"]
    )

    # Approximate relative humidity
    F["rh_proxy"] = (
        100 - 5 * F["dewp_depress"]
    )

    # Simple measure of conditions that may help disperse pollution
    F["vent"] = (
        F["WSPM_i_o0"] *
        (F["TEMP_i_o0"] - F["TEMP_i_o-4"])
    )

    # Calendar information for the target hour
    h = grid.ts + pd.Timedelta("1h")

    F["hour"] = h.dt.hour.astype("int16")
    F["dow"] = h.dt.dayofweek.astype("int16")
    F["month"] = h.dt.month.astype("int16")

    doy = h.dt.dayofyear.astype("float32")

    # Cyclical features preserve the repeating nature of hours and seasons
    F["doy_sin"] = np.sin(
        2 * np.pi * doy / 365.25
    ).astype("float32")

    F["doy_cos"] = np.cos(
        2 * np.pi * doy / 365.25
    ).astype("float32")

    F["hr_sin"] = np.sin(
        2 * np.pi * F["hour"] / 24
    ).astype("float32")

    F["hr_cos"] = np.cos(
        2 * np.pi * F["hour"] / 24
    ).astype("float32")

    # Beijing's heating season tends to have different pollution behaviour
    F["is_heating"] = (
        h.dt.month
        .isin([11, 12, 1, 2, 3])
        .astype("int8")
    )

    # Keep the original sensor values at the target hour as well as the
    # interpolated versions above
    for c in NUMS:
        F["raw_" + c] = (
            g[c]
            .shift(-1)
            .astype("float32")
        )

    # Number of missing measurements at the target and observation hours
    F["n_missing_h"] = (
        g["n_missing"]
        .shift(-1)
        .astype("float32")
    )

    F["n_missing_t"] = (
        grid["n_missing"]
        .astype("float32")
    )

    # Categorical information for station and wind direction
    F["station"] = grid.station.astype("category")

    F["wd_h"] = (
        g["wd"]
        .shift(-1)
        .astype("category")
    )

    F["wd_t"] = grid["wd"].astype("category")

    # Keep these columns so the data can later be split back into train/test
    # and used for chronological validation
    F["ts"] = grid.ts.values
    F["is_test"] = grid.is_test.values
    F["id"] = grid.id.values
    F["y"] = grid.PM2_5_next_hour.values

    return F


def build_all():
    # Run the full feature engineering process

    tr, te = load()

    # Reconstruct each station's full hourly timeline
    grid = build_grid(tr, te)

    # Generate all model features
    F = make_features(grid)

    # Remove artificial rows that were only inserted to maintain hourly spacing
    F = F[F.id.notna()].reset_index(drop=True)

    return F


# Build the feature table
F = build_all()

print(F.shape)

# Save it so later modelling cells do not need to rebuild the features
F.to_pickle("features.pkl")

print("saved")

## 3. Feature Set v2

This version expands the original feature set to capture more temporal and spatial information.

Compared with v1, it:

- uses a wider window around the target hour, from `-8` to `+3`
- adds PM10, CO and NO2 readings from each of the 12 monitoring stations at the target hour
- includes a longer 48-hour rolling history
- adds city-wide rolling averages and minimum/maximum values
- includes a pressure trend feature

These additions allow the model to better capture longer-term pollution patterns and differences between individual monitoring stations across Beijing.

In [ ]:
import numpy as np
import pandas as pd

# Folder containing the training and test data
UPLOADS = "."

# Pollutant measurements
POLL = ["PM10", "SO2", "NO2", "CO", "O3"]

# Weather measurements
MET = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

# All numerical sensor variables
NUMS = POLL + MET

# Hours around the target hour that will be used as features
OFFSETS = (-8, -6, -4, -3, -2, -1, 0, 1, 2, 3)

# Convert compass wind directions into numerical angles
WD_DEG = {"N": 0, "NNE": 22.5, "NE": 45, "ENE": 67.5, "E": 90, "ESE": 112.5,
          "SE": 135, "SSE": 157.5, "S": 180, "SSW": 202.5, "SW": 225,
          "WSW": 247.5, "W": 270, "WNW": 292.5, "NW": 315, "NNW": 337.5}


def build_grid():

    # Load the labelled training data and the competition test data
    tr = pd.read_csv(f"{UPLOADS}/train.csv", parse_dates=["observation_timestamp"])
    te = pd.read_csv(f"{UPLOADS}/test.csv", parse_dates=["observation_timestamp"])

    # Mark which rows originally came from train and test
    tr["is_test"], te["is_test"] = 0, 1

    # Test targets are unknown, so add an empty target column before combining
    te["PM2_5_next_hour"] = np.nan

    # Combine train and test because they form one continuous timeline
    full = pd.concat([tr, te], ignore_index=True).rename(
        columns={"observation_timestamp": "ts"})

    # Find all stations in the dataset
    stations = sorted(full.station.unique())

    # Create every hourly timestamp across the full train and test period
    hours = pd.date_range(full.ts.min(), full.ts.max(), freq="h")

    # Build a complete timeline for every station so time shifts always
    # correspond to real clock hours
    grid = pd.MultiIndex.from_product([stations, hours],
                                      names=["station", "ts"]).to_frame(index=False)

    # Add the actual observations onto the complete hourly grid
    grid = grid.merge(full, on=["station", "ts"], how="left")
    grid = grid.sort_values(["station", "ts"]).reset_index(drop=True)

    # Convert wind direction and wind speed into two numerical wind components
    rad = np.deg2rad(grid.wd.map(WD_DEG))
    grid["wu"] = -grid.WSPM * np.sin(rad)
    grid["wv"] = -grid.WSPM * np.cos(rad)

    # Count how many sensor measurements are missing at each station and hour
    grid["n_missing"] = grid[NUMS].isna().sum(axis=1)

    return grid, stations


def make_features(grid, stations):

    # Include the original numerical sensors and the two wind components
    vcols = NUMS + ["wu", "wv"]

    # Group observations by station so interpolation does not mix stations
    g0 = grid.groupby("station", sort=False)

    # Fill short missing gaps within each station before creating temporal features
    ip = pd.DataFrame({c + "_i": g0[c].transform(
        lambda x: x.interpolate(limit=8, limit_direction="both")).astype("float32")
        for c in vcols}, index=grid.index)

    # Add the interpolated versions back onto the main grid
    grid = pd.concat([grid, ip], axis=1)

    # Keep track of the names of the interpolated columns
    icols = [c + "_i" for c in vcols]

    # Group by station again for the lag and lead calculations
    g = grid.groupby("station", sort=False)

    # Store the engineered features here before creating the final DataFrame
    feats = {}

    # Create measurements from several hours before and after the target hour
    for c in icols:
        for k in OFFSETS:
            feats[f"{c}_o{k}"] = g[c].shift(-(k + 1)).astype("float32")

        # Measure how each variable has changed around the target hour
        feats[f"{c}_d1"] = feats[f"{c}_o0"] - feats[f"{c}_o-1"]
        feats[f"{c}_d3"] = feats[f"{c}_o0"] - feats[f"{c}_o-3"]
        feats[f"{c}_d8"] = feats[f"{c}_o0"] - feats[f"{c}_o-8"]

        # These capture what happens shortly after the target hour
        feats[f"{c}_dfwd"] = feats[f"{c}_o1"] - feats[f"{c}_o0"]
        feats[f"{c}_dfwd3"] = feats[f"{c}_o3"] - feats[f"{c}_o0"]

    # Add rolling statistics to describe recent pollution and weather history
    for c in ["PM10_i", "CO_i", "NO2_i", "SO2_i", "O3_i", "WSPM_i", "TEMP_i", "DEWP_i"]:

        # Align each variable with the target hour
        s = g[c].shift(-1).groupby(grid.station, sort=False)

        # Average conditions over several recent time windows
        for w in (6, 12, 24, 48):
            feats[f"{c}_rm{w}"] = s.transform(
                lambda x, w=w: x.rolling(w, min_periods=2).mean()).astype("float32")

        # Measure short and longer term variability
        feats[f"{c}_rs12"] = s.transform(
            lambda x: x.rolling(12, min_periods=3).std()).astype("float32")
        feats[f"{c}_rs48"] = s.transform(
            lambda x: x.rolling(48, min_periods=6).std()).astype("float32")

        # Record recent highs and lows
        feats[f"{c}_rmax12"] = s.transform(
            lambda x: x.rolling(12, min_periods=3).max()).astype("float32")
        feats[f"{c}_rmin12"] = s.transform(
            lambda x: x.rolling(12, min_periods=3).min()).astype("float32")

        # Compare the target hour measurement with its recent average
        feats[f"{c}_vs_rm24"] = feats[f"{c}_o0"] - feats[f"{c}_rm24"]
        feats[f"{c}_vs_rm48"] = feats[f"{c}_o0"] - feats[f"{c}_rm48"]

    # Convert the collected temporal features into a DataFrame
    F = pd.DataFrame(feats, index=grid.index)

    # Build city wide summaries of pollution and weather at each timestamp
    # aggregates at the target hour
    ncols = ["PM10_i", "CO_i", "NO2_i", "SO2_i", "O3_i", "WSPM_i", "TEMP_i", "PRES_i", "DEWP_i"]
    net_src = grid[["ts"] + ncols]
    gt = net_src.groupby("ts")

    # Average each measurement across all monitoring stations
    net = gt.mean()
    net.columns = ["net_" + c for c in net.columns]

    # The target hour is one hour after the timestamp of the original row
    h_idx = grid.ts + pd.Timedelta("1h")

    # City wide conditions at the target hour, observation hour and six hours earlier
    at_h = net.reindex(h_idx).to_numpy(dtype="float32")
    at_h1 = net.reindex(grid.ts).to_numpy(dtype="float32")
    at_h6 = net.reindex(grid.ts - pd.Timedelta("5h")).to_numpy(dtype="float32")

    # Store the city wide level and recent change for every network variable
    for j, name in enumerate(net.columns):
        F[name] = at_h[:, j]
        F[name + "_d1"] = at_h[:, j] - at_h1[:, j]
        F[name + "_d6"] = at_h[:, j] - at_h6[:, j]

    # Compare each individual station with the overall city conditions
    for c in ["PM10_i", "CO_i", "NO2_i", "O3_i"]:
        F[f"anom_{c}"] = F[f"{c}_o0"] - F[f"net_{c}"]
        F[f"ratio_{c}"] = F[f"{c}_o0"] / (F[f"net_{c}"] + 1e-3)

    # Add the spread, minimum and maximum PM10 and CO levels across the city
    for stat in ("std", "min", "max"):
        v = getattr(gt[["PM10_i", "CO_i"]], stat)().reindex(h_idx).to_numpy(dtype="float32")
        F[f"net_PM10_{stat}"], F[f"net_CO_{stat}"] = v[:, 0], v[:, 1]

    # Track the recent city wide PM10 trend
    # network rolling mean of PM10 (regional trend, ending at h)
    npm = net["net_PM10_i"]
    for w in (12, 24):
        F[f"net_PM10_rm{w}"] = npm.rolling(w, min_periods=3).mean().reindex(
            h_idx).to_numpy(dtype="float32")

    # Give the model the individual readings from every station rather than
    # representing the whole city using only averages
    # Lets the model read spatial gradients (upwind vs downwind) rather than
    # collapsing the network to a single mean.
    for var in ["PM10_i", "CO_i", "NO2_i"]:

        # Reshape the data so each monitoring station becomes its own column
        wide = grid.pivot_table(index="ts", columns="station", values=var)

        # Retrieve all station readings at the target hour
        w_at_h = wide.reindex(h_idx).to_numpy(dtype="float32")

        # Add a separate feature for each station
        for j, st in enumerate(wide.columns):
            F[f"st_{var}_{st}"] = w_at_h[:, j]

    # Add relationships between pollutants and weather conditions
    # Create pollutant ratio and weather features
    F["PM10_over_CO"] = F["PM10_i_o0"] / (F["CO_i_o0"] + 1.0)
    F["NO2_over_CO"] = F["NO2_i_o0"] / (F["CO_i_o0"] + 1.0)
    F["SO2_over_NO2"] = F["SO2_i_o0"] / (F["NO2_i_o0"] + 1.0)
    F["O3_over_NO2"] = F["O3_i_o0"] / (F["NO2_i_o0"] + 1.0)

    # Temperature minus dew point provides a simple measure related to humidity
    F["dewp_depress"] = F["TEMP_i_o0"] - F["DEWP_i_o0"]

    # City wide version of the same humidity related measure
    F["dewp_depress_net"] = F["net_TEMP_i"] - F["net_DEWP_i"]

    # Approximate relative humidity
    F["rh_proxy"] = 100 - 5 * F["dewp_depress"]

    # Combine wind speed and temperature change as a rough measure of ventilation
    F["vent"] = F["WSPM_i_o0"] * (F["TEMP_i_o0"] - F["TEMP_i_o-4"])

    # Measure how atmospheric pressure has changed over the previous eight hours
    F["pres_trend"] = F["PRES_i_o0"] - F["PRES_i_o-8"]

    # Add information about the time and season of the target hour
    # Add calendar information for the target hour
    F["hour"] = h_idx.dt.hour.astype("int16")
    F["dow"] = h_idx.dt.dayofweek.astype("int16")
    F["month"] = h_idx.dt.month.astype("int16")

    doy = h_idx.dt.dayofyear.astype("float32")

    # Encode the yearly cycle using sine and cosine so December and January
    # are treated as being close together
    F["doy_sin"] = np.sin(2 * np.pi * doy / 365.25).astype("float32")
    F["doy_cos"] = np.cos(2 * np.pi * doy / 365.25).astype("float32")

    # Do the same for the repeating 24 hour daily cycle
    F["hr_sin"] = np.sin(2 * np.pi * F["hour"] / 24).astype("float32")
    F["hr_cos"] = np.cos(2 * np.pi * F["hour"] / 24).astype("float32")

    # Mark whether the target hour occurs during Beijing's heating season
    F["is_heating"] = h_idx.dt.month.isin([11, 12, 1, 2, 3]).astype("int8")

    # Keep the original uninterpolated sensor readings at the target hour
    for c in NUMS:
        F["raw_" + c] = g[c].shift(-1).astype("float32")

    # Keep track of how many measurements are missing at the target and observation hours
    F["n_missing_h"] = g["n_missing"].shift(-1).astype("float32")
    F["n_missing_t"] = grid["n_missing"].astype("float32")

    # Keep station and wind direction as categorical features for LightGBM
    F["station"] = grid.station.astype("category")
    F["wd_h"] = g["wd"].shift(-1).astype("category")
    F["wd_t"] = grid["wd"].astype("category")

    # Keep the original timestamp, train/test indicator, ID and target so they
    # can be used later for validation and prediction
    F["ts"], F["is_test"] = grid.ts.values, grid.is_test.values
    F["id"], F["y"] = grid.id.values, grid.PM2_5_next_hour.values

    return F


# Build the complete hourly grid
# Build the final feature set
grid, stations = build_grid()

# Generate all v2 features
F = make_features(grid, stations)

# Remove artificial grid rows that were only added to preserve hourly spacing
F = F[F.id.notna()].reset_index(drop=True)

# Check the final feature table size
print(F.shape)

# Save the features so they can be reused without rebuilding them
F.to_pickle("features2.pkl")
print("saved")

## 4. Time Aware Validation

Because the dataset follows a chronological timeline, the model should be validated on future observations rather than randomly selected rows.

The model is trained using all observations before September 2015 and then evaluated on data from September 2015 to February 2016. This validation period is particularly useful because it covers the same months as the competition test set.

A random split could give an overly optimistic RMSE because observations close together in time have very similar pollution conditions and share information through the time based features. Keeping the validation period completely after the training period gives a more realistic estimate of how well the model will perform on unseen future data.

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb, sys

# Load the engineered v2 features created in the previous section
F = pd.read_pickle("features2.pkl")

# These columns are needed for bookkeeping but should not be used as model inputs
DROP = ["ts", "is_test", "id", "y"]

# Use every remaining column as a predictor
FEATS = [c for c in F.columns if c not in DROP]

# Tell LightGBM which variables should be treated as categorical
CATS = ["station", "wd_h", "wd_t"]

# Keep only the labelled training observations
trn = F[F.is_test == 0].reset_index(drop=True)

# Save the validation results to a log file
log = open("v2.log", "a", buffering=1)

# LightGBM settings used for the v2 model
P = dict(objective="regression", metric="rmse", learning_rate=0.04,
         num_leaves=127, min_data_in_leaf=60, feature_fraction=0.45,
         bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, lambda_l1=0.5,
         max_bin=255, num_threads=-1, verbosity=-1, seed=42)

# Use September 2015 to February 2016 as the validation period
# This matches the months covered by the competition test set
vs, ve = "2015-09-01", "2016-03-01"

# Train only on observations that occurred before the validation period
a = trn[trn.ts < vs]

# Use the following six months as unseen validation data
b = trn[(trn.ts >= vs) & (trn.ts < ve)]

# Convert the training data into LightGBM's dataset format
da = lgb.Dataset(a[FEATS], a.y.values, categorical_feature=CATS)

# Create the validation dataset using exactly the same features
db = lgb.Dataset(b[FEATS], b.y.values, categorical_feature=CATS, reference=da)

# Train the model and stop if validation performance does not improve
# for 150 consecutive boosting rounds
bst = lgb.train(P, da, num_boost_round=3000, valid_sets=[db],
                callbacks=[lgb.early_stopping(150, verbose=False)])

# Generate validation predictions using the best number of boosting rounds
# Predictions are limited to the valid PM2.5 range
pv = np.clip(bst.predict(b[FEATS], num_iteration=bst.best_iteration), 2, 999)

# Calculate RMSE on the unseen validation period
r = float(np.sqrt(np.mean((b.y.values - pv) ** 2)))

# Record the v2 RMSE so it can be compared with the previous feature set
print(f"V2 features: iter={bst.best_iteration} RMSE={r:.4f}   (v1 was 21.3934)", file=log)

# Save the validation predictions for later analysis or model comparisons
np.save("pv_v2.npy", pv)

# Save the validation IDs and true PM2.5 values
b[["id", "y"]].to_pickle("va_v1block.pkl")

# Save the feature importance scores so we can see which variables
# contributed most to the model
pd.Series(bst.feature_importance("gain"), index=FEATS).sort_values(
    ascending=False).to_csv("imp2.csv")

# Record that the validation run completed successfully
print("DONE", file=log)

## 5. Error Analysis

This section looks at which observations contribute most to the model's validation error.

Instead of treating every mistake as equally important, the true PM2.5 values are divided into ten groups from lowest to highest. The squared prediction error is then calculated for each group.

This shows whether most of the RMSE is coming from ordinary pollution levels or from a smaller number of severe pollution events. In this case, the highest PM2.5 decile contributes a large share of the total squared error, which suggests that improving predictions during extreme pollution events is likely to have the biggest effect on the final RMSE.

In [ ]:
# Load the engineered v2 features
F2 = pd.read_pickle("features2.pkl")

# Keep only the labelled training rows
trn = F2[F2.is_test == 0].reset_index(drop=True)

# Recreate the same validation period used in the previous section
b = trn[(trn.ts >= "2015-09-01") & (trn.ts < "2016-03-01")].reset_index(drop=True)

# Load the saved validation predictions and the corresponding true PM2.5 values
p = np.load("pv_v2.npy")
y = b.y.values

# Calculate the squared error for every validation observation
se = (y - p) ** 2

# Split the true PM2.5 values into ten equally sized groups
# Decile 0 contains the lowest pollution values and decile 9 the highest
dec = pd.qcut(y, 10, labels=False)

# Calculate how much each PM2.5 decile contributes to the total squared error
share = pd.Series(se).groupby(dec).sum() / se.sum()

# Show the share of total error coming from each pollution level
print("share of total squared error by TRUE decile:")
print(share.round(3).to_string())

# Highlight how much of the total error comes from the most polluted 10 percent
print("\ntop decile share:", round(share.iloc[-1], 3))

share of total squared error by TRUE decile:
0    0.018
1    0.006
2    0.009
3    0.013
4    0.017
5    0.026
6    0.030
7    0.047
8    0.084
9    0.749

top decile share: 0.749


## 6. Not Calibrating the tail

The model looks badly miscalibrated at the top: on the 2015-16 holdout the top
predicted decile averages 289 where truth averages 324. Fitting an isotonic
correction on Sep–Nov and testing on Dec–Feb cuts RMSE from 24.6 to 20.1.

It is not free. Fit the same correction on the **previous** winter and apply it
to this one and RMSE goes from 20.95 to **26.74**:

| | winter 2014-15 | winter 2015-16 |
|---|---|---|
| top-decile bias (true − pred) | **−22.1** (over-predicts) | **+34.3** (under-predicts) |

The bias is not a stable property of the model but rather it's a year-specific
distribution shift, and its sign flips. The 24.6 -> 20.1 gain was only reachable
because the correction was fitted on the same winter it was scored on, which is
impossible for the test winter. **No calibration goes into the submission.**

This also reframes what's worth optimising: between-winter shift is the
dominant error source, not model capacity, which argues for blend robustness
over sharpness.

In [ ]:
"""Does a tail calibration fitted on winter N transfer to winter N+1?

This mirrors what we'd be doing at submission time: fit the correction on the
most recent labelled winter (Sep15-Feb16) and apply it to a *different* winter
predicted by a model trained on *more* data (the test period, Sep16-Feb17).

Analogue tested here:
  fit  : model trained <=Aug 2014, predicting Sep14-Feb15  -> fit calibration
  apply: model trained <=Aug 2015, predicting Sep15-Feb16  -> does it help?
If yes, the same procedure should transfer one year further to the test set.
"""
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.isotonic import IsotonicRegression

# Load features and keep only the labelled rows
F = pd.read_pickle("features2.pkl")
DROP = ["ts", "is_test", "id", "y"]
FEATS = [c for c in F.columns if c not in DROP]
CATS = ["station", "wd_h", "wd_t"]
trn = F[F.is_test == 0].reset_index(drop=True)

log = open("calib.log", "a", buffering=1)
rmse = lambda y, p: float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

# Same LightGBM params used throughout
P = dict(objective="regression", metric="rmse", learning_rate=0.04,
         num_leaves=127, min_data_in_leaf=60, feature_fraction=0.45,
         bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, lambda_l1=0.5,
         max_bin=255, num_threads=-1, verbosity=-1, seed=42)

# Earlier winter: train <= Aug 2014, predict Sep14-Feb15
a = trn[trn.ts < "2014-09-01"]
b = trn[(trn.ts >= "2014-09-01") & (trn.ts < "2015-03-01")]
bst = lgb.train(P, lgb.Dataset(a[FEATS], a.y.values, categorical_feature=CATS),
                num_boost_round=291)
p_prev = np.clip(bst.predict(b[FEATS]), 2, 999)
y_prev = b.y.values
print(f"winter 2014-15: RMSE {rmse(y_prev, p_prev):.4f}  n={len(b)}", file=log)

# Look at the decile-level bias on this winter
q = pd.qcut(p_prev, 10, labels=False)
d = pd.DataFrame({"p": p_prev, "y": y_prev, "q": q}).groupby("q").agg(
    pred=("p", "mean"), true=("y", "mean"))
print("decile bias (true - pred), winter 2014-15:", file=log)
print((d.true - d.pred).round(2).to_string(), file=log)

# Fit calibrations on that winter
iso = IsotonicRegression(out_of_bounds="clip").fit(p_prev, y_prev)
lin = np.polyfit(p_prev, y_prev, 1)
print(f"linear fit on 2014-15: slope={lin[0]:.4f} intercept={lin[1]:.2f}", file=log)

# Apply to the NEXT winter, predicted by a model trained on more data
p_next = np.load("pv_v2.npy")          # Sep15-Feb16, model <=Aug 2015
y_next = trn[(trn.ts >= "2015-09-01") & (trn.ts < "2016-03-01")].y.values
print(f"\nwinter 2015-16 (the real holdout), n={len(y_next)}", file=log)
print(f"  uncalibrated           {rmse(y_next, p_next):.4f}", file=log)
print(f"  isotonic from 2014-15  {rmse(y_next, np.clip(iso.predict(p_next), 2, 999)):.4f}",
      file=log)
print(f"  linear   from 2014-15  {rmse(y_next, np.clip(np.polyval(lin, p_next), 2, 999)):.4f}",
      file=log)

# Also try a shrunk version of the isotonic correction
for s in (0.25, 0.5, 0.75):
    pc = np.clip((1 - s) * p_next + s * np.clip(iso.predict(p_next), 2, 999), 2, 999)
    print(f"  isotonic shrunk x{s:.2f}   {rmse(y_next, pc):.4f}", file=log)

print("DONE", file=log)

## 7. Final models and the blend

Trained on all labelled data. `num_boost_round` is fixed at ~1.25x the validated
`best_iteration` — the full training set is larger than the validation fold's,
and there's no honest holdout left to early-stop on. Three seeds averaged.
Predictions clipped to the observed target range [2, 999].

In [ ]:
# Load the v1 feature matrix and define the feature / categorical columns
F = pd.read_pickle("features.pkl")
DROP = ["ts", "is_test", "id", "y"]
FEATS = [c for c in F.columns if c not in DROP]
CATS = ["station", "wd_h", "wd_t"]

# Split into train and test once
trn = F[F.is_test == 0].reset_index(drop=True)
tst = F[F.is_test == 1].reset_index(drop=True)
samp = pd.read_csv("./sample_submission.csv")

# Log progress so we can watch the multi-seed run
log = open("final.log", "a", buffering=1)

# Shared LightGBM settings (tuned on the validation fold)
BASE = dict(objective="regression", metric="rmse", learning_rate=0.04,
            num_leaves=127, min_data_in_leaf=60, feature_fraction=0.55,
            bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, lambda_l1=0.5,
            max_bin=255, num_threads=-1, verbosity=-1)

N_ROUNDS = 600          # 473 best_iter on validation, scaled 1.25x for the larger full-train set
preds = []

# Train three independent models with different seeds and keep a running average
for seed in (7, 202, 1337):
    p = dict(BASE, seed=seed, bagging_seed=seed + 1, feature_fraction_seed=seed + 2)
    dtr = lgb.Dataset(trn[FEATS], trn.y.values, categorical_feature=CATS)
    bst = lgb.train(p, dtr, num_boost_round=N_ROUNDS)
    preds.append(bst.predict(tst[FEATS]))

    # Average so far → clip to a sensible range → write submission
    pred = np.clip(np.mean(preds, axis=0), 2, 999)
    sub = samp[["id"]].merge(pd.DataFrame({"id": tst.id.values,
                                           "PM2_5_next_hour": pred}), on="id", how="left")
    assert sub.PM2_5_next_hour.notna().all() and len(sub) == len(samp)
    sub.to_csv("submission_v1.csv", index=False)

    print(f"seed {seed}: submission updated ({len(preds)} models averaged), "
          f"mean={pred.mean():.2f} min={pred.min():.2f} max={pred.max():.2f}", file=log)

    # Save feature importance from the first seed only (for later inspection)
    if seed == 7:
        pd.Series(bst.feature_importance("gain"), index=FEATS
                  ).sort_values(ascending=False).to_csv("importance.csv")

print("DONE", file=log)

In [ ]:
# Load the v1 feature matrix
F = pd.read_pickle("features.pkl")
DROP = ["ts", "is_test", "id", "y"]
FEATS = [c for c in F.columns if c not in DROP]
CATS = ["station", "wd_h", "wd_t"]

# Split into train / test
trn = F[F.is_test == 0].reset_index(drop=True)
tst = F[F.is_test == 1].reset_index(drop=True)
samp = pd.read_csv("./sample_submission.csv")

# Log file so we can monitor the multi-seed run
log = open("final.log", "a", buffering=1)

# Base LightGBM parameters (tuned earlier on the validation fold)
BASE = dict(objective="regression", metric="rmse", learning_rate=0.04,
            num_leaves=127, min_data_in_leaf=60, feature_fraction=0.55,
            bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, lambda_l1=0.5,
            max_bin=255, num_threads=-1, verbosity=-1)

N_ROUNDS = 600          # 473 best_iter on validation, scaled 1.25x for the larger full-train set
preds = []

# Train three models with different seeds and keep a running average
for seed in (7, 202, 1337):
    p = dict(BASE, seed=seed, bagging_seed=seed + 1, feature_fraction_seed=seed + 2)
    dtr = lgb.Dataset(trn[FEATS], trn.y.values, categorical_feature=CATS)
    bst = lgb.train(p, dtr, num_boost_round=N_ROUNDS)
    preds.append(bst.predict(tst[FEATS]))

    # Average so far → clip → write submission
    pred = np.clip(np.mean(preds, axis=0), 2, 999)
    sub = samp[["id"]].merge(pd.DataFrame({"id": tst.id.values,
                                           "PM2_5_next_hour": pred}), on="id", how="left")
    assert sub.PM2_5_next_hour.notna().all() and len(sub) == len(samp)
    sub.to_csv("submission_v1.csv", index=False)

    print(f"seed {seed}: submission updated ({len(preds)} models averaged), "
          f"mean={pred.mean():.2f} min={pred.min():.2f} max={pred.max():.2f}", file=log)

    # Dump feature importance from the first seed only
    if seed == 7:
        pd.Series(bst.feature_importance("gain"), index=FEATS
                  ).sort_values(ascending=False).to_csv("importance.csv")

print("DONE", file=log)

Blend weights swept on the holdout — optimum is flat between 0.7 and 0.8.

In [ ]:
# v1 validation -> produces pv_v1_raw.npy, needed by the blend cell below.
F1 = pd.read_pickle("features.pkl")
FE1 = [c for c in F1.columns if c not in ["ts","is_test","id","y"]]
CATS = ["station","wd_h","wd_t"]

# Keep only the labelled (train) rows
t1 = F1[F1.is_test==0].reset_index(drop=True)

# Season-matched hold-out: train on everything before Sep 2015,
# validate on the following winter/spring block
a = t1[t1.ts < "2015-09-01"]; b = t1[(t1.ts>="2015-09-01")&(t1.ts<"2016-03-01")]

# Build LightGBM datasets (db references da so categorical handling stays consistent)
da = lgb.Dataset(a[FE1], a.y.values, categorical_feature=CATS)
db = lgb.Dataset(b[FE1], b.y.values, categorical_feature=CATS, reference=da)

# Same params used later for the full-data retrain
P1 = dict(objective="regression", metric="rmse", learning_rate=0.04, num_leaves=127,
          min_data_in_leaf=60, feature_fraction=0.55, bagging_fraction=0.8, bagging_freq=1,
          lambda_l2=5.0, lambda_l1=0.5, max_bin=255, num_threads=-1, verbosity=-1, seed=42)

# Train with early stopping on the hold-out block
m1 = lgb.train(P1, da, num_boost_round=3000, valid_sets=[db],
               callbacks=[lgb.early_stopping(150, verbose=False)])

# Predict on the hold-out, clip to a sensible range, and save
pv1 = np.clip(m1.predict(b[FE1], num_iteration=m1.best_iteration), 2, 999)
np.save("pv_v1_raw.npy", pv1)
b[["id","y"]].to_pickle("va_v1block.pkl")

print(f"v1 holdout: iter={m1.best_iteration} RMSE={rmse(b.y.values, pv1):.4f}")   # 21.39

v1 holdout: iter=516 RMSE=21.7508


In [ ]:
# Load the two hold-out prediction vectors and the corresponding true targets
p1 = np.load("pv_v1_raw.npy"); p2 = np.load("pv_v2.npy")
y = pd.read_pickle("va_v1block.pkl").y.values

# Quick grid search over blend weights
for w in [0, .2, .4, .5, .6, .7, .72, .8, 1.0]:
    print(f"  w_v2={w:.2f}  RMSE={rmse(y, (1 - w) * p1 + w * p2):.4f}")

# Build the final test-set blend using the weight that looked best on the hold-out
v2p = pd.read_pickle("pred_v2_ids.pkl").rename(columns={"PM2_5_next_hour": "v2"})
v1p = pd.read_csv("submission_v1.csv").rename(columns={"PM2_5_next_hour": "v1"})
m = v1p.merge(v2p, on="id"); assert m.isna().sum().sum() == 0

out = pd.DataFrame({"id": m.id,
                    "PM2_5_next_hour": np.clip(0.72 * m.v2 + 0.28 * m.v1, 2, 999)})

# Align with the official sample submission order and write the file
samp = pd.read_csv(f"{UPLOADS}/sample_submission.csv")
out = samp[["id"]].merge(out, on="id", how="left")
assert out.PM2_5_next_hour.notna().all() and len(out) == len(samp)
out.to_csv("submission_blend.csv", index=False)
print(out.shape, round(out.PM2_5_next_hour.mean(), 2))

  w_v2=0.00  RMSE=21.7508
  w_v2=0.20  RMSE=21.4767
  w_v2=0.40  RMSE=21.2761
  w_v2=0.50  RMSE=21.2040
  w_v2=0.60  RMSE=21.1510
  w_v2=0.70  RMSE=21.1172
  w_v2=0.72  RMSE=21.1128
  w_v2=0.80  RMSE=21.1027
  w_v2=1.00  RMSE=21.1319
(51063, 2) 93.12


## 8. Variant W — capacity aimed at severe events

Since ~75% of the error sits in the top true decile, weight those rows up.

The weights are a function of **PM10 at the target hour — a feature, not the
label.** That distinction matters: weighting by `y` would bias the fitted
conditional mean upward and hurt RMSE, whereas weighting by a covariate only
reallocates capacity and leaves `E[y|x]` unbiased.

Alone it is slightly worse than v2 (21.06 vs 20.95) but it decorrelates, and the
three-way blend reaches **20.8752** at v1=0.20, v2=0.55, W=0.25.

In [ ]:
import gc

# Append-mode log so we can resume / monitor long runs
log = open("resumeW.log", "a", buffering=1)
say = lambda *a: print(*a, file=log)

# LightGBM params tuned earlier; slightly higher num_leaves + regularisation
# for the full-data retrain
P = dict(objective="regression", metric="rmse", learning_rate=0.04,
         num_leaves=127, min_data_in_leaf=60, feature_fraction=0.45,
         bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, lambda_l1=0.5,
         max_bin=255, num_threads=-1, verbosity=-1, seed=11,
         bagging_seed=12, feature_fraction_seed=13)
N_ROUNDS = 422                      # 338 best_iter × 1.25 for the larger full-train set
W1, W2, WW = 0.20, 0.55, 0.25       # blend weights fitted on the season-matched holdout

# Load the big feature matrix (v2)
F = pd.read_pickle("features2.pkl")
DROP = ["ts", "is_test", "id", "y"]
FEATS = [c for c in F.columns if c not in DROP]
CATS = ["station", "wd_h", "wd_t"]

# Spill the test rows to disk first so we can free the whole frame
tst_mask = F.is_test == 1
F.loc[tst_mask, FEATS + ["id"]].reset_index(drop=True).to_pickle("tstX.pkl")
say(f"test features spilled ({int(tst_mask.sum())} rows)")

# Keep only training rows in memory
trn = F.loc[~tst_mask].reset_index(drop=True)
del F
gc.collect()

# Sample weights: up-weight high-PM10 hours (they matter more for RMSE)
pm = trn.PM10_i_o0.fillna(trn.net_PM10_i).fillna(80.0).values
w = np.clip(1.0 + pm / 120.0, 1.0, 4.0)

# Build the LightGBM dataset and free the pandas frame
dtr = lgb.Dataset(trn[FEATS], trn.y.values, categorical_feature=CATS, weight=w)
dtr.construct()
del trn, pm, w
gc.collect()
say("dataset constructed, training")

# Train on everything (no early stopping – fixed round count)
bst = lgb.train(P, dtr, num_boost_round=N_ROUNDS)
del dtr
gc.collect()
say("trained")

# Predict on the held-out test features
tstX = pd.read_pickle("tstX.pkl")
predW = bst.predict(tstX[FEATS])
ids = tstX.id.values
del tstX, bst
gc.collect()
np.save("pred_W.npy", predW)
say(f"predicted, mean={predW.mean():.2f}")

# 3-way blend: v1 + v2 + this “W” model
v2p = pd.read_pickle("pred_v2_ids.pkl").rename(columns={"PM2_5_next_hour": "v2"})
v1p = pd.read_csv("submission_v1.csv").rename(columns={"PM2_5_next_hour": "v1"})
df = pd.DataFrame({"id": ids, "W": predW}).merge(v2p, on="id").merge(v1p, on="id")
assert len(df) == len(ids) and df.isna().sum().sum() == 0

# Clip to a sensible physical range
df["PM2_5_next_hour"] = np.clip(W1 * df.v1 + W2 * df.v2 + WW * df.W, 2, 999)

# Align with the official sample submission order
samp = pd.read_csv("./sample_submission.csv")
out = samp[["id"]].merge(df[["id", "PM2_5_next_hour"]], on="id", how="left")
assert out.PM2_5_next_hour.notna().all() and len(out) == len(samp)
out.to_csv("submission_3way.csv", index=False)
say(f"3-way blend written: {out.shape}, mean={out.PM2_5_next_hour.mean():.2f}")
say("DONE")